# SolarSDE — Final Publication Notebook

**Code provenance:** every stage below pulls and runs the actual modules from [github.com/keshavkrishnan08/SDE](https://github.com/keshavkrishnan08/SDE) — nothing is rewritten or embedded.

**What this notebook produces (one GPU run):**
1. Full SKIPP'D pipeline (517 days, VAE + optical-flow motion + CTI)
2. **Both architectures** trained: closed-form Mixture-of-OU *and* Euler-Maruyama latent rollout
3. **Ensemble** of the two + **champion selection on validation** (never test)
4. SkyGPT exact-benchmark (identical Nov–Dec 2019 cloudy test) for **all variants**, full 1–30 min band
5. Baselines, ablations, stratified+DM, leave-one-month-out CV, PIT/bootstrap, ramp AUROC, CTI validation, multi-level reliability, sampling efficiency, compute cost, Holm-Bonferroni, CAISO economics + sensitivity, figures, LaTeX tables

Every stage is failure-isolated: an error prints and the run continues to the final zip.

*Honesty note: the head-to-head vs SkyGPT (CRPS 2.81 at h=15, their cloudy test) is reported exactly as measured — whichever way it comes out.*

## 0. Environment

In [ ]:
# ==== Setup: environment, directories, torch warmup ====
import os, sys, json, math, time, gc, shutil, subprocess, traceback
from pathlib import Path
import numpy as np, pandas as pd

# torch._dynamo warmup (some Kaggle builds crash on lazy import at optimizer creation)
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
# CUDA_LAUNCH_BLOCKING makes device-side asserts raise at the offending op (not a
# later CUBLAS call), so a crash is attributed to the real cause and isolated to
# its own stage instead of surfacing mysteriously downstream.
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
import torch
try:
    import torch._utils, torch._dynamo  # noqa: F401
except Exception as _e:
    print(f"[WARN] dynamo warmup: {_e} — continuing")
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kaggle={IN_KAGGLE} Colab={IN_COLAB} device={DEVICE}")
if DEVICE.type != "cuda":
    print("[WARN] No GPU — enable a GPU runtime. Training both architectures on CPU is impractical.")

ROOT = (Path("/kaggle/working") if IN_KAGGLE else Path.cwd()) / "final_run"
PERSIST_DIR = ROOT / "outputs"; WORK_DIR = ROOT / "work"; DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"; RESULTS_DIR = PERSIST_DIR / "results"
LATENT_DIR = PERSIST_DIR / "latents"; SPLITS_DIR = PERSIST_DIR / "splits"
EXTENDED_DIR = PERSIST_DIR / "extended"; FIGURES_DIR = PERSIST_DIR / "figures"
for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, LATENT_DIR, SPLITS_DIR, EXTENDED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===== Run configuration (the only knobs you may want to touch) =====
Z_DIM = 64
SKIPPD_VAE_EPOCHS  = 12     # CS-VAE epochs
CLOSEDFORM_EPOCHS  = 60     # closed-form SDE training epochs
ROLLOUT_EPOCHS     = 35     # rollout SDE epochs (each epoch costs ~3-4x closed-form)
CV_EPOCHS          = 15     # cross-validation epochs per fold
CV_MAX_FOLDS       = 6
print(f"PERSIST_DIR={PERSIST_DIR}")
print(f"Config: VAE={SKIPPD_VAE_EPOCHS}ep, closed-form={CLOSEDFORM_EPOCHS}ep, "
      f"rollout={ROLLOUT_EPOCHS}ep, CV={CV_EPOCHS}ep x {CV_MAX_FOLDS} folds")


## 1. Pull the codebase from GitHub (the repo code IS the experiment code)

In [ ]:
# ==== Pull the SolarSDE codebase from GitHub and import the actual modules ====
# The code that runs below IS the repo code (github.com/keshavkrishnan08/SDE),
# not a copy embedded in this notebook.
REPO_HTTPS = "https://github.com/keshavkrishnan08/SDE.git"
REPO_ZIP   = "https://github.com/keshavkrishnan08/SDE/archive/refs/heads/main.zip"
REPO_DIR = ROOT / "sde_repo"

def _clone_repo():
    if (REPO_DIR / "notebooks" / "_solarsde_v2.py").exists():
        print(f"  repo already present at {REPO_DIR}")
        # refresh to latest main (best effort)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                       capture_output=True, timeout=120)
        return True
    for attempt in range(1, 4):
        try:
            print(f"  git clone (attempt {attempt}) ...")
            r = subprocess.run(["git", "clone", "--depth", "1", REPO_HTTPS, str(REPO_DIR)],
                               capture_output=True, text=True, timeout=300)
            if r.returncode == 0 and (REPO_DIR / "notebooks").exists():
                return True
            print(f"    clone failed: {r.stderr[:200]}")
        except Exception as e:
            print(f"    clone error: {e}")
        time.sleep(5)
    # Fallback: download the repo as a zip archive
    try:
        print("  falling back to zip archive download ...")
        import urllib.request, zipfile, io
        with urllib.request.urlopen(REPO_ZIP, timeout=300) as r:
            zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extractall(ROOT)
        extracted = next(ROOT.glob("SDE-*"))
        if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
        extracted.rename(REPO_DIR)
        return (REPO_DIR / "notebooks").exists()
    except Exception as e:
        print(f"    zip fallback failed: {e}")
        return False

if not _clone_repo():
    raise RuntimeError("Could not obtain the SolarSDE repo from GitHub — check network/repo access.")
MODULE_DIR = REPO_DIR / "notebooks"
sys.path.insert(0, str(MODULE_DIR))
print(f"  modules dir: {MODULE_DIR}")
print(f"  repo modules: {sorted(p.name for p in MODULE_DIR.glob('_*.py'))}")

# ---- Fallback-guarded imports: a missing/broken module never stops the run ----
def _safe_import(module, names):
    out = {}
    try:
        mod = __import__(module, fromlist=names)
        for n in names:
            out[n] = getattr(mod, n)
        print(f"  [OK]   {module}: {len(names)} constants")
    except Exception as e:
        print(f"  [FAIL] {module}: {type(e).__name__}: {str(e)[:120]}")
        for n in names:
            out[n] = f'print("[SKIP] {n} unavailable — module {module} failed to import")'
    return out

globals().update(_safe_import("_master_hardening", ["safe_stage"]))
globals().update(_safe_import("_combined_generator",
    ["SHARED_CODE", "BASELINES_CODE", "STRATIFIED_CODE", "ANALYSIS_CODE"]))
globals().update(_safe_import("_final_generator",
    ["LOAD_DATA_TOLERANT_CODE", "RAMP_AUROC_CODE", "BOOTSTRAP_CIS_CODE",
     "PIT_RELIABILITY_CODE", "ECONOMIC_CAISO_CODE", "LATEX_TABLES_CODE", "ZIP_DOWNLOAD_CODE"]))
globals().update(_safe_import("_colab_master_generator",
    ["CTI_VALIDATION_CODE", "HOLM_BONFERRONI_CODE"]))
globals().update(_safe_import("_skippd_pipeline",
    ["SKIPPD_DOWNLOAD_FULL_CODE", "SKIPPD_PREP_CODE", "SKIPPD_VAE_CODE",
     "SKIPPD_LATENTS_WRITE_CODE", "SKIPPD_HORIZON_OVERRIDE_CODE"]))
globals().update(_safe_import("_solarsde_v2",
    ["MDN_ARCHITECTURE_CODE", "STAGE_0_V2_CODE", "POST_STAGE0_V2_VERIFY_CODE", "ABLATIONS_V2_CODE"]))
globals().update(_safe_import("_solarsde_rollout",
    ["ROLLOUT_ARCH_CODE", "ABLATIONS_ROLLOUT_CODE"]))
globals().update(_safe_import("_ensemble_eval",
    ["STASH_CLOSEDFORM_CODE", "STASH_ROLLOUT_CODE", "CHAMPION_SELECT_CODE",
     "SKYGPT_TRIPLE_BENCHMARK_CODE"]))
globals().update(_safe_import("_skippd_extras",
    ["IMPLEMENTATION_DETAILS_CODE", "DATA_CARD_CODE", "COMPUTATIONAL_COST_CODE",
     "RELIABILITY_LEVELS_CODE", "SAMPLING_EFFICIENCY_CODE", "ECONOMIC_SENSITIVITY_CODE",
     "CROSS_VALIDATION_V2_CODE"]))

# If safe_stage itself failed to import, provide a minimal local fallback.
if isinstance(globals().get("safe_stage"), str):
    def safe_stage(name, code):
        ind = "\n".join("    " + l if l else "" for l in code.splitlines())
        return (f"try:\n{ind}\nexcept Exception as _e:\n"
                f"    import traceback; traceback.print_exc()\n"
                f"    print('[STAGE FAILED] {name} — continuing.')\n")
    print("  [WARN] using local fallback safe_stage")
print("\nAll modules wired. Code provenance: github.com/keshavkrishnan08/SDE @ main")


## 2. Download all data — SKIPP'D (~2.3 GB) + SkyGPT exact test set

In [ ]:
# ==== DOWNLOAD_SKIPPD ====
try:
    exec(safe_stage('DOWNLOAD_SKIPPD', SKIPPD_DOWNLOAD_FULL_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] DOWNLOAD_SKIPPD — continuing to next cell.')


## 3. Preprocess — clear-sky-PV, ramps, chronological splits

In [ ]:
# ==== SKIPPD_PREP ====
try:
    exec(safe_stage('SKIPPD_PREP', SKIPPD_PREP_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_PREP — continuing to next cell.')


## 4. CS-VAE (64×64 → 64-d) + encode all frames + optical-flow motion features

In [ ]:
# ==== SKIPPD_VAE ====
try:
    exec(safe_stage('SKIPPD_VAE', SKIPPD_VAE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_VAE — continuing to next cell.')


## 5. CTI + write the {splits, extended, latents} contract

In [ ]:
# ==== SKIPPD_WRITE ====
try:
    exec(safe_stage('SKIPPD_WRITE', SKIPPD_LATENTS_WRITE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_WRITE — continuing to next cell.')


## 6. Shared metrics + load tensors + 1-min horizon config

In [ ]:
# ==== SHARED ====
try:
    exec(safe_stage('SHARED', SHARED_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SHARED — continuing to next cell.')


In [ ]:
# ==== LOAD_DATA ====
try:
    exec(safe_stage('LOAD_DATA', LOAD_DATA_TOLERANT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] LOAD_DATA — continuing to next cell.')


In [ ]:
# ==== HORIZON_OVERRIDE ====
try:
    exec(safe_stage('HORIZON_OVERRIDE', SKIPPD_HORIZON_OVERRIDE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] HORIZON_OVERRIDE — continuing to next cell.')


## 6a. Data card + implementation details (reproducibility)

In [ ]:
# ==== DATA_CARD ====
try:
    exec(safe_stage('DATA_CARD', DATA_CARD_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] DATA_CARD — continuing to next cell.')


In [ ]:
# ==== IMPLEMENTATION_DETAILS ====
try:
    exec(safe_stage('IMPLEMENTATION_DETAILS', IMPLEMENTATION_DETAILS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] IMPLEMENTATION_DETAILS — continuing to next cell.')


## 7. ARCHITECTURE A — Closed-form Mixture-of-OU: train + calibrate + evaluate

In [ ]:
# ==== CLOSEDFORM_ARCH ====
try:
    exec(safe_stage('CLOSEDFORM_ARCH', MDN_ARCHITECTURE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_ARCH — continuing to next cell.')


In [ ]:
# ==== CLOSEDFORM_GLUE ====
try:
    # Keep a named reference to the closed-form class before the rollout
    # architecture overwrites the TemporalLatentSDE alias.
    ClosedFormSDE = TemporalLatentSDE
    print('ClosedFormSDE alias saved.')
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_GLUE — continuing to next cell.')


In [ ]:
# ==== CLOSEDFORM_TRAIN ====
try:
    exec(safe_stage('STAGE0_CLOSEDFORM',
         STAGE_0_V2_CODE.replace('EPOCHS = 60', f'EPOCHS = {CLOSEDFORM_EPOCHS}')), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_TRAIN — continuing to next cell.')


In [ ]:
# ==== CLOSEDFORM_VERIFY ====
try:
    exec(safe_stage('CLOSEDFORM_VERIFY', POST_STAGE0_V2_VERIFY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_VERIFY — continuing to next cell.')


In [ ]:
# ==== STASH_CLOSEDFORM ====
try:
    exec(safe_stage('STASH_CLOSEDFORM', STASH_CLOSEDFORM_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] STASH_CLOSEDFORM — continuing to next cell.')


## 8. ARCHITECTURE B — Euler-Maruyama latent rollout: train + calibrate + evaluate

In [ ]:
# ==== ROLLOUT_ARCH ====
try:
    exec(safe_stage('ROLLOUT_ARCH', ROLLOUT_ARCH_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ROLLOUT_ARCH — continuing to next cell.')


In [ ]:
# ==== ROLLOUT_TRAIN ====
try:
    exec(safe_stage('STAGE0_ROLLOUT',
         STAGE_0_V2_CODE.replace('EPOCHS = 60', f'EPOCHS = {ROLLOUT_EPOCHS}')), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ROLLOUT_TRAIN — continuing to next cell.')


In [ ]:
# ==== ROLLOUT_VERIFY ====
try:
    exec(safe_stage('ROLLOUT_VERIFY', POST_STAGE0_V2_VERIFY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ROLLOUT_VERIFY — continuing to next cell.')


In [ ]:
# ==== STASH_ROLLOUT ====
try:
    exec(safe_stage('STASH_ROLLOUT', STASH_ROLLOUT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] STASH_ROLLOUT — continuing to next cell.')


## 9. Champion selection (on VALIDATION) — closed-form vs rollout vs ensemble

In [ ]:
# ==== CHAMPION_SELECT ====
try:
    exec(safe_stage('CHAMPION_SELECT', CHAMPION_SELECT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CHAMPION_SELECT — continuing to next cell.')


## 10. SkyGPT EXACT BENCHMARK — all variants, identical cloudy test, full 1–30 min band

In [ ]:
# ==== SKYGPT_TRIPLE_BENCHMARK ====
try:
    exec(safe_stage('SKYGPT_TRIPLE_BENCHMARK', SKYGPT_TRIPLE_BENCHMARK_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKYGPT_TRIPLE_BENCHMARK — continuing to next cell.')


## 11. Ablations (champion-matched: closed-form or rollout native)

In [ ]:
# ==== ABLATIONS ====
try:
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    _abl = ABLATIONS_V2_CODE if globals().get('CHAMPION_SINGLE', 'closedform') == 'closedform' \
           else ABLATIONS_ROLLOUT_CODE
    exec(safe_stage('ABLATIONS', _abl), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ABLATIONS — continuing to next cell.')


## 12. Leave-one-month-out cross-validation

In [ ]:
# ==== CROSS_VALIDATION ====
try:
    exec(safe_stage('CROSS_VALIDATION', CROSS_VALIDATION_V2_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CROSS_VALIDATION — continuing to next cell.')


## 13. Sampling efficiency + computational cost

In [ ]:
# ==== SAMPLING_EFFICIENCY ====
try:
    exec(safe_stage('SAMPLING_EFFICIENCY', SAMPLING_EFFICIENCY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SAMPLING_EFFICIENCY — continuing to next cell.')


In [ ]:
# ==== COMPUTATIONAL_COST ====
try:
    exec(safe_stage('COMPUTATIONAL_COST', COMPUTATIONAL_COST_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] COMPUTATIONAL_COST — continuing to next cell.')


## 14. Stratified eval + Diebold-Mariano significance

In [ ]:
# ==== STRATIFIED ====
try:
    exec(safe_stage('STRATIFIED', STRATIFIED_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] STRATIFIED — continuing to next cell.')


## 15. PIT / reliability + bootstrap CIs

In [ ]:
# ==== PIT_RELIABILITY ====
try:
    exec(safe_stage('PIT_RELIABILITY', PIT_RELIABILITY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] PIT_RELIABILITY — continuing to next cell.')


In [ ]:
# ==== BOOTSTRAP_CIS ====
try:
    exec(safe_stage('BOOTSTRAP_CIS', BOOTSTRAP_CIS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] BOOTSTRAP_CIS — continuing to next cell.')


## 16. Ramp AUROC + CTI physical validation

In [ ]:
# ==== RAMP_AUROC ====
try:
    exec(safe_stage('RAMP_AUROC', RAMP_AUROC_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] RAMP_AUROC — continuing to next cell.')


In [ ]:
# ==== CTI_VALIDATION ====
try:
    exec(safe_stage('CTI_VALIDATION', CTI_VALIDATION_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CTI_VALIDATION — continuing to next cell.')


## 17. Multi-level reliability

In [ ]:
# ==== RELIABILITY_LEVELS ====
try:
    exec(safe_stage('RELIABILITY_LEVELS', RELIABILITY_LEVELS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] RELIABILITY_LEVELS — continuing to next cell.')


## 18. Holm-Bonferroni + CAISO economics + sensitivity

In [ ]:
# ==== HOLM_BONFERRONI ====
try:
    exec(safe_stage('HOLM_BONFERRONI', HOLM_BONFERRONI_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] HOLM_BONFERRONI — continuing to next cell.')


In [ ]:
# ==== ECONOMIC_CAISO ====
try:
    exec(safe_stage('ECONOMIC_CAISO', ECONOMIC_CAISO_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ECONOMIC_CAISO — continuing to next cell.')


In [ ]:
# ==== ECONOMIC_SENSITIVITY ====
try:
    exec(safe_stage('ECONOMIC_SENSITIVITY', ECONOMIC_SENSITIVITY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ECONOMIC_SENSITIVITY — continuing to next cell.')


## 19. Baselines (persistence, smart-persistence, LSTM, MC-Dropout, CSDI)

In [ ]:
# ==== BASELINES ====
try:
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    exec(safe_stage('BASELINES', BASELINES_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] BASELINES — continuing to next cell.')


## 20. Analysis figures + LaTeX tables

In [ ]:
# ==== ANALYSIS ====
try:
    exec(safe_stage('ANALYSIS', ANALYSIS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ANALYSIS — continuing to next cell.')


In [ ]:
# ==== LATEX_TABLES ====
try:
    exec(safe_stage('LATEX_TABLES', LATEX_TABLES_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] LATEX_TABLES — continuing to next cell.')


## Final — Zip the complete paper package

In [ ]:
# ==== ZIP ====
try:
    exec(safe_stage('ZIP', ZIP_DOWNLOAD_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ZIP — continuing to next cell.')
